# Ensembl Gene ID to Gene Symbol/Name Converter

This version fixes the issue where MyGene.info may say an ID is found but still returns no gene symbol/name.

It uses multiple sources:

1. **Ensembl REST lookup** for `display_name`, `description`, and `biotype`
2. **MyGene.info** as a backup
3. Optional Excel input/output

For Ensembl IDs like `ENSG000002...`, some may be non-coding, withdrawn, or poorly annotated, so they may not have a normal gene symbol.

In [ ]:
!pip -q install pandas openpyxl requests mygene

In [ ]:
import time
import requests
import pandas as pd
import mygene

gene_ids = [
    "ENSG00000174111",
    "ENSG00000238683",
    "ENSG00000130723",
    "ENSG00000254184",
    "ENSG00000255322",
    "ENSG00000262370",
    "ENSG00000233864",
    "ENSG00000205664",
    "ENSG00000241860",
    "ENSG00000260661",
    "ENSG00000272373",
    "ENSG00000243491",
]

# Remove version numbers if your IDs look like ENSG00000123456.7
gene_ids = [str(x).strip().split('.')[0] for x in gene_ids if pd.notna(x)]
gene_ids

## Function 1: Query Ensembl REST

This is usually better than MyGene.info for Ensembl IDs because it returns Ensembl's own `display_name`, `description`, and `biotype`.

In [ ]:
def query_ensembl_lookup(gene_id, server="https://rest.ensembl.org"):
    """Query Ensembl REST lookup/id endpoint for one Ensembl gene ID."""
    url = f"{server}/lookup/id/{gene_id}"
    headers = {"Content-Type": "application/json"}
    params = {"expand": 0}

    try:
        r = requests.get(url, headers=headers, params=params, timeout=20)
        if r.status_code != 200:
            return {
                "ensembl_gene_id": gene_id,
                "ensembl_found": False,
                "ensembl_symbol": None,
                "ensembl_name": None,
                "ensembl_biotype": None,
                "ensembl_status": r.status_code,
            }

        data = r.json()
        return {
            "ensembl_gene_id": gene_id,
            "ensembl_found": True,
            "ensembl_symbol": data.get("display_name"),
            "ensembl_name": data.get("description"),
            "ensembl_biotype": data.get("biotype"),
            "ensembl_status": r.status_code,
        }

    except Exception as e:
        return {
            "ensembl_gene_id": gene_id,
            "ensembl_found": False,
            "ensembl_symbol": None,
            "ensembl_name": None,
            "ensembl_biotype": None,
            "ensembl_status": f"ERROR: {e}",
        }


ensembl_rows = []
for gid in gene_ids:
    ensembl_rows.append(query_ensembl_lookup(gid))
    time.sleep(0.1)  # be polite to the server

ensembl_df = pd.DataFrame(ensembl_rows)
ensembl_df

## Function 2: Query MyGene.info as backup

MyGene.info sometimes returns the Ensembl ID as found but does not return the symbol. That is why the final result below prioritizes Ensembl first.

In [ ]:
mg = mygene.MyGeneInfo()

mygene_results = mg.querymany(
    gene_ids,
    scopes="ensembl.gene",
    fields="symbol,name,type_of_gene,entrezgene",
    species="human",
    as_dataframe=False,
    returnall=False,
    verbose=False
)

mygene_rows = []
for item in mygene_results:
    mygene_rows.append({
        "ensembl_gene_id": item.get("query"),
        "mygene_found": not item.get("notfound", False),
        "mygene_symbol": item.get("symbol"),
        "mygene_name": item.get("name"),
        "mygene_gene_type": item.get("type_of_gene"),
        "mygene_entrez_id": item.get("entrezgene"),
        "mygene_id": item.get("_id"),
    })

mygene_df = pd.DataFrame(mygene_rows)
mygene_df

## Combine results and choose the best available name

In [ ]:
final_df = ensembl_df.merge(mygene_df, on="ensembl_gene_id", how="left")

# Prefer Ensembl display_name first, then MyGene symbol
final_df["best_gene_symbol"] = final_df["ensembl_symbol"].combine_first(final_df["mygene_symbol"])

# Prefer Ensembl description first, then MyGene name
final_df["best_gene_name_or_description"] = final_df["ensembl_name"].combine_first(final_df["mygene_name"])

# Prefer Ensembl biotype first, then MyGene type
final_df["best_gene_type"] = final_df["ensembl_biotype"].combine_first(final_df["mygene_gene_type"])

display_cols = [
    "ensembl_gene_id",
    "best_gene_symbol",
    "best_gene_name_or_description",
    "best_gene_type",
    "ensembl_found",
    "mygene_found",
    "mygene_entrez_id",
]

final_df[display_cols]

## Save output files

In [ ]:
final_df.to_excel("ensembl_gene_id_mapping_FIXED.xlsx", index=False)
final_df.to_csv("ensembl_gene_id_mapping_FIXED.csv", index=False)

print("Saved:")
print("- ensembl_gene_id_mapping_FIXED.xlsx")
print("- ensembl_gene_id_mapping_FIXED.csv")

## Optional: read gene IDs from your own Excel file

Use this if your IDs are already in an Excel sheet. Put the file in the same folder as the notebook, then edit the file name and column name.

In [ ]:
# Example only. Uncomment and edit when needed.

# input_file = "your_excel_file.xlsx"
# sheet_name = "Sheet1"
# column_name = "GeneID"   # Change this to your actual column header

# input_df = pd.read_excel(input_file, sheet_name=sheet_name)
# gene_ids = input_df[column_name].dropna().astype(str).str.strip().str.split('.').str[0].tolist()
# gene_ids[:10]

## Notes

If an ID still has no symbol/name after this:

- it may be retired/deprecated
- it may be from an older Ensembl release
- it may be a non-coding or predicted locus
- it may not have a standard HGNC gene symbol

For old Ensembl IDs, you may need to search the Ensembl archive matching the annotation version used by your dataset.